# Structural trapping: spill-point analysis

This notebook reproduces the structural-trapping example of Section 3 of
`report/report.tex`. It runs the geometric spill-point analysis of MRST's
`co2lab` module on a synthetic sandbox aquifer, without solving any flow
equation.

Before dynamic simulation, the shape of the caprock already sets a bound on
how much CO2 a formation can trap structurally. The `co2lab` module builds a
top-surface grid from the 3-D grid and analyzes its depth function. A
**trap** is the interior of a closed depth contour around a local minimum
(a dome apex). Its **spill point** is the saddle where CO2 overflows into
the next trap. The traps and the spill paths that connect them form a tree
that predicts the long-term migration of buoyant CO2, with no flow solve.

This example is adapted from `co2lab-spillpoint/examples/firstTrappingExample.m`.
Two changes make it run headless on this machine:

- `trapAnalysis` uses the edge-based method (`false`). The cell-based method
  needs `matlab_bgl` MEX binaries, which Octave cannot load.
- Figures are 2-D depth/trap maps. This example never builds a PEBI mesh, so
  it needs neither the `upr` module nor the `fix/upr-octave-block-comments`
  branch.

Runtime: well under a minute.

In [ ]:
run('/home/adriano/codes/MRST/startup.m');
mrstModule add co2lab-spillpoint co2lab-common co2lab-legacy co2lab-demo coarsegrid
mrstVerbose off

FIGDIR = '/home/adriano/codes/MRST/reproductions/salo2024-gcs/figures';

## Synthetic sandbox aquifer

A $10\times5$ km, 50 m thick sandbox at 1 km depth, with a dipping,
sinusoidally perturbed top surface: $100\times100$ top-surface cells,
porosity 0.25. `topSurfaceGrid` extracts the 2-D caprock surface grid from
the 3-D volumetric grid.

In [ ]:
[Lx,Ly,H] = deal(10000, 5000, 50);
[nx,ny]   = deal(100, 100);
G  = cartGrid([nx ny 1],[Lx Ly H]);
x  = G.nodes.coords(1:G.nodes.num/2,1)/Lx;
y  = G.nodes.coords(1:G.nodes.num/2,2)/Ly;
z  = G.nodes.coords(1:G.nodes.num/2,3)/H;
zt = z + x - 0.2*sin(5*pi*x).*sin(5*pi*y.^1.5) - 0.075*sin(1.25*pi*y) + 0.15*sin(x+y);
zb = 1 + x;
G.nodes.coords(:,3) = [zt; zb]*H+1000;
G = computeGeometry(G);

Gt = topSurfaceGrid(G);
fprintf('Top-surface grid: %d cells\n', Gt.cells.num);

## Trap analysis

`trapAnalysis` finds every local depth minimum bounded by a closed contour
and labels the cells inside it with a trap index.

In [ ]:
res = trapAnalysis(Gt, false);
num_traps = max(res.traps);
fprintf('Number of traps found: %d\n', num_traps);

### Depth map with trap cells

White dots mark cells inside a structural trap. Compare with the left panel
of Fig. 1 in the report.

In [ ]:
xc = Gt.cells.centroids(:,1);
yc = Gt.cells.centroids(:,2);
zmap = reshape(Gt.cells.z, nx, ny)';
figure;
imagesc([min(xc) max(xc)], [min(yc) max(yc)], zmap);
axis xy equal tight; colormap(jet); colorbar;
title('Top surface depth (m); white = trap cells');
hold on
tmask = reshape(double(res.traps>0), nx, ny)';
[yy,xx] = find(tmask);
plot((xx-0.5)*Lx/nx, (yy-0.5)*Ly/ny, 'w.', 'MarkerSize', 4);
hold off
print(gcf, fullfile(FIGDIR, 'fig1_depth_and_traps.png'), '-dpng', '-r120');

### Traps and spill paths

Each trap is numbered; CO2 that overflows a trap migrates along its spill
path (green) to the next trap up-dip. Compare with the right panel of
Fig. 1 in the report.

In [ ]:
trap_field = zeros(size(res.traps));
trap_field(res.traps>0) = 2;
for r = [res.cell_lines{:}]'
    for c = 1:numel(r)
        trap_field(r{c}) = 1;
    end
end

figure;
imagesc([min(xc) max(xc)], [min(yc) max(yc)], reshape(trap_field, nx, ny)');
axis xy equal tight; colormap(jet);
title('Traps (red), spill paths (green), other cells (blue)');
for i=1:num_traps
   ind = res.top(i);
   text(Gt.cells.centroids(ind,1), Gt.cells.centroids(ind,2), ...
      num2str(res.traps(ind)), 'Color', .99*[1 1 1], ...
      'FontSize', 12, 'HorizontalAlignment', 'center');
end
print(gcf, fullfile(FIGDIR, 'fig2_traps_and_spillpaths.png'), '-dpng', '-r120');

## Trapping capacity

The pore volume of every trap, summed, gives the total structural trapping
capacity of this aquifer.

In [ ]:
rock2D.poro = 0.25 * ones(G.cells.num, 1);
trap_volumes = volumesOfTraps(Gt, res, 1:num_traps, 'poro', rock2D.poro);

total_trapping_capacity = sum(trap_volumes);
pv = sum(poreVolume(Gt,rock2D));
fprintf('Total trapping capacity is: %6.3e\n', total_trapping_capacity);
fprintf('This amounts to %.2f %% of a total pore volume of %6.2e\n\n', ...
   total_trapping_capacity/pv*100, pv);
[sorted_vols, sorted_ix] = sort(trap_volumes, 'descend');

fprintf('trap ix   | trap vol(m3)  | cells in trap\n');
fprintf('----------+---------------+--------------\n');
tcells = zeros(num_traps,1);
for i = 1:num_traps
   tcells(i) = sum(res.traps == sorted_ix(i));
   fprintf('%7d   |  %10.3e   | %5d\n', ...
       sorted_ix(i), sorted_vols(i), tcells(i));
end
fprintf(['\nTogether, the five largest traps cover %6.2e m3, which represents %3.1f%% of' ...
   '\nthe total trapping capacity of this grid.\n'], ...
   sum(sorted_vols(1:5)), sum(sorted_vols(1:5)) / total_trapping_capacity * 100)

### Trap volumes, sorted

In [ ]:
figure;
subplot(2,1,1);
bar(1:num_traps, sorted_vols);
set(gca,'XTick',1:num_traps,'XTickLabel',sorted_ix);
title('Trap volume (m^3), sorted');
subplot(2,1,2);
bar(1:num_traps, tcells);
set(gca,'XTick',1:num_traps,'XTickLabel',sorted_ix);
title('Number of cells in trap');
print(gcf, fullfile(FIGDIR, 'fig3_trap_volumes.png'), '-dpng', '-r120');